# Multimodal Experiment Runner

Run a complete experiment end-to-end, unattended, on Colab compute. Clones the repo, builds the multimodal dataset, checks modality independence, trains ablation variants via purged walk-forward CV, runs a portfolio backtest, and saves all artifacts to a timestamped folder on Google Drive.

Expected runtime on Colab GPU: ~30â€“60 minutes for the default 6-stock / 1-year config. Larger universes (50 stocks / 5 years) take 2â€“4 hours on CPU.

*(For a step-by-step walkthrough of the architecture and data pipeline, see the demo notebook â€” in progress.)*

In [ ]:
import os, sys

REPO_URL  = 'https://github.com/Randhir123/nifty50-multimodal-transformer.git'
REPO_NAME = 'nifty50-multimodal-transformer'

try:
    from google.colab import drive as _colab_check
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    repo_root = f'/content/{REPO_NAME}'
    if not os.path.exists(repo_root):
        ret = os.system(f'git clone {REPO_URL} {repo_root}')
        if ret != 0:
            raise RuntimeError('git clone failed â€” check repo URL and network')
    os.chdir(repo_root)
    os.system('pip install -r requirements.txt -q')
    os.system('pip install -e . -q')
else:
    # Running locally â€” assume CWD is already the repo root
    repo_root = os.getcwd()

sys.path.insert(0, repo_root)
print(f'CWD      : {os.getcwd()}')
print(f'IN_COLAB : {IN_COLAB}')

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted.')
else:
    print('Not on Colab â€” outputs go to ./experiments')

In [ ]:
# === EXPERIMENT CONFIG ===
# Edit these. All other cells pick up changes from here automatically.

TICKERS = "RELIANCE.NS,TCS.NS,INFY.NS,HDFCBANK.NS,ICICIBANK.NS,SBIN.NS"
# Comma-separated NSE tickers. Default is the 6-stock smoke set.
# For full Nifty 50 expansion, paste the full list here.

PERIOD = "1y"
# yfinance period string: "1y", "2y", "5y", "10y", or "max".
# Longer periods exercise more market regimes but need more compute.

WINDOW_SIZE = 20
HORIZON_DAYS = 3
# OHLCV window length and forward-return horizon. Don't change unless
# you know why â€” these match the leakage-test assumptions.

EPOCHS = 20
BATCH_SIZE = 16
# Training schedule per ablation variant per CV fold.

CV_SPLITS = 3
EMBARGO_DAYS = 3
# Walk-forward purged CV configuration.

OUTPUT_ROOT = "/content/drive/MyDrive/nifty50_experiments" if IN_COLAB else "./experiments"
# Where artifacts are written. A timestamped subdirectory is created.

DEVICE = "cuda"  # set to "cpu" if no GPU

In [ ]:
import sys
import time
import traceback
import json
import subprocess
import yfinance as yf
from datetime import date, timedelta
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display, Image, Markdown

# Hard-coded training configuration â€” do not change these between runs
SEED = 42
LEARNING_RATE = 1e-3
WEIGHT_DECAY  = 1e-4
# Multi-seed evaluation is out of scope for this unattended runner.

# Timestamped output directory
timestamp = time.strftime("%Y%m%d_%H%M%S")
run_id    = f"exp_{timestamp}"
output_dir = Path(OUTPUT_ROOT) / run_id
output_dir.mkdir(parents=True, exist_ok=True)

# Canonical paths â€” defined here so all cells always have them
dataset_path = output_dir / "real_world_multimodal_samples_gaf.npz"
raw_dir      = output_dir / "raw"

print(f"Run ID  : {run_id}")
print(f"Output  : {output_dir}")
print(f"Device  : {DEVICE}")

# Summary written at the end regardless of which steps fail
summary_lines = [
    f"# Experiment Summary: {run_id}\n",
    "## Configuration",
    f"- Tickers : {TICKERS}",
    f"- Period  : {PERIOD}",
    f"- Window  : {WINDOW_SIZE}d  Horizon: {HORIZON_DAYS}d",
    f"- Epochs  : {EPOCHS}  Batch: {BATCH_SIZE}",
    f"- CV      : {CV_SPLITS} splits  Embargo: {EMBARGO_DAYS}d",
    f"- Device  : {DEVICE}",
    f"- Output  : {output_dir}\n",
    "## Execution Log",
]
start_time = time.time()

In [ ]:
print("Step 1: Building multimodal artifact...")
ticker_list = [t.strip() for t in TICKERS.split(",") if t.strip()]
try:
    subprocess.run(
        [
            sys.executable, "scripts/run_real_world_demo.py",
            "--tickers", *ticker_list,
            "--period", PERIOD,
            "--window-size", str(WINDOW_SIZE),
            "--horizon-days", str(HORIZON_DAYS),
            "--output-dir", str(output_dir),
            "--device", DEVICE,
        ],
        check=True,
        cwd=repo_root,
    )
    summary_lines.append("- Data Prep: SUCCESS")
    print("Done.")
except subprocess.CalledProcessError as e:
    print(f"ERROR in data prep (script exited with code {e.returncode})")
    summary_lines.append(f"- Data Prep: FAILED (exit code {e.returncode})")
except Exception as e:
    print(f"ERROR in data prep: {e}")
    traceback.print_exc()
    summary_lines.append(f"- Data Prep: FAILED ({e})")

In [ ]:
if dataset_path.exists():
    _d  = np.load(dataset_path, allow_pickle=False)
    n   = len(_d['y'])
    pos = float(_d['y'].mean())
    print(f'Samples       : {n}')
    print(f'Positive rate : {pos:.1%}')
    for k in _d.files:
        if _d[k].ndim > 0:
            print(f'  {k}: {_d[k].shape}')
    summary_lines += ['\n## Artifact',
                       f'- Samples: {n}',
                       f'- Positive rate: {pos:.1%}']
    del _d
else:
    print('Artifact not found â€” data prep may have failed. Check errors above.')

In [ ]:
print("Step 2: Modality independence check...")
try:
    if dataset_path.exists():
        indep_csv = output_dir / 'modality_independence.csv'
        subprocess.run(
            [
                sys.executable, 'scripts/check_modality_independence.py',
                '--artifact', str(dataset_path),
                '--output-csv', str(indep_csv),
            ],
            check=True,
            cwd=repo_root,
        )
        if indep_csv.exists():
            df_i = pd.read_csv(indep_csv, index_col=0)
            display(df_i.round(4))
            summary_lines += ['\n## Modality Independence',
                               df_i.round(4).to_markdown()]
        summary_lines.append('- Independence Check: SUCCESS')
except Exception as e:
    print(f'ERROR in independence check: {e}')
    summary_lines.append(f'- Independence Check: FAILED ({e})')

In [ ]:
print("Step 3: Running ablations (long step)...")
# Known gap: the ablation runner has no mid-fold resume.
# A Colab disconnect mid-training requires restarting from fold 0.
try:
    if dataset_path.exists():
        subprocess.run(
            [
                sys.executable, 'scripts/run_ablation_study.py',
                '--dataset', str(dataset_path),
                '--output-dir', str(output_dir / 'ablations'),
                '--epochs', str(EPOCHS),
                '--batch-size', str(BATCH_SIZE),
                '--device', DEVICE,
                '--cv-splits', str(CV_SPLITS),
                '--horizon-days', str(HORIZON_DAYS),
                '--embargo-days', str(EMBARGO_DAYS),
            ],
            check=True,
            cwd=repo_root,
        )
        summary_lines.append('- Ablations: SUCCESS')
except Exception as e:
    print(f'ERROR in ablations: {e}')
    summary_lines.append(f'- Ablations: FAILED ({e})')

In [ ]:
try:
    abl_csv = output_dir / 'ablations' / 'ablation_results.csv'
    if abl_csv.exists():
        df_ab = pd.read_csv(abl_csv)
        cols  = [c for c in ['variant', 'roc_auc_mean', 'accuracy_mean', 'f1_mean']
                 if c in df_ab.columns]
        display(df_ab[cols])
        summary_lines += ['\n## Ablation Results',
                           df_ab[cols].to_markdown(index=False)]
    else:
        print('No ablation results found.')
except Exception as e:
    print(f'Could not display ablation results: {e}')

In [ ]:
print("Step 4: Running backtest...")
try:
    # Prefer best-performing single-addition variant; fall back to tabular_only
    preds_csv = output_dir / 'ablations' / 'prediction_scores_tabular_image.csv'
    if not preds_csv.exists():
        preds_csv = output_dir / 'ablations' / 'prediction_scores_tabular_only.csv'
    tab_csv = output_dir / 'tabular_samples.csv'

    if preds_csv.exists() and tab_csv.exists():
        subprocess.run(
            [
                sys.executable, 'scripts/run_backtest.py',
                '--predictions', str(preds_csv),
                '--tabular-samples', str(tab_csv),
                '--raw-dir', str(raw_dir),
                '--output-dir', str(output_dir / 'backtest'),
                '--top-k', '1',
                '--horizon-days', str(HORIZON_DAYS),
            ],
            check=True,
            cwd=repo_root,
        )
        curve = output_dir / 'backtest' / 'backtest_curve.png'
        if curve.exists():
            display(Image(filename=str(curve)))
        metrics_p = output_dir / 'backtest' / 'backtest_metrics.json'
        if metrics_p.exists():
            m = json.loads(metrics_p.read_text())
            summary_lines.append('\n## Backtest')
            for k, v in m.items():
                summary_lines.append(f'- {k}: {v}')
        summary_lines.append('- Backtest: SUCCESS')
    else:
        print('Prediction scores not found â€” skipping backtest.')
        summary_lines.append('- Backtest: SKIPPED (predictions not found)')
except Exception as e:
    print(f'ERROR in backtest: {e}')
    summary_lines.append(f'- Backtest: FAILED ({e})')

In [ ]:
elapsed = (time.time() - start_time) / 60
summary_lines.append(f'\n**Total runtime**: {elapsed:.1f} minutes')

summary_path = output_dir / 'summary.md'
summary_path.write_text('\n'.join(summary_lines), encoding='utf-8')

print(f'Done in {elapsed:.1f} minutes.')
print(f'All artifacts at: {output_dir}')
display(Markdown(f'Results saved to `{output_dir}`'))